In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

In [ ]:
import os

for dirname, subdirs, filenames in os.walk("/kaggle/input"):
    print(dirname)

In [ ]:
DATASET_PATH = "/kaggle/input/human-skin-diseases-image/SkinDisease"

In [ ]:
import os
import tensorflow as tf

# Search for train, valid, and test folders automatically
TRAIN_DIR = None
VALID_DIR = None
TEST_DIR = None

for root, dirs, files in os.walk("/kaggle/input"):
    if os.path.basename(root).lower() == "train":
        TRAIN_DIR = root

    elif os.path.basename(root).lower() in ["valid", "validation"]:
        VALID_DIR = root

    elif os.path.basename(root).lower() == "test":
        TEST_DIR = root

print("TRAIN PATH:", TRAIN_DIR)
print("VALID PATH:", VALID_DIR)
print("TEST PATH:", TEST_DIR)

# Check that paths were found
if TRAIN_DIR is None:
    raise FileNotFoundError("Train folder was not found!")

if VALID_DIR is None:
    raise FileNotFoundError("Valid folder was not found!")

if TEST_DIR is None:
    raise FileNotFoundError("Test folder was not found!")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

valid_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_dataset.class_names

print("Classes:")
print(class_names)

print("Number of Classes:", len(class_names))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

valid_dataset = valid_dataset.prefetch(
    buffer_size=AUTOTUNE
)

test_dataset = test_dataset.prefetch(
    buffer_size=AUTOTUNE
)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

In [ ]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

In [ ]:
model = models.Sequential([
    
    layers.Input(shape=(224, 224, 3)),
    
    data_augmentation,
    
    base_model,
    
    layers.GlobalAveragePooling2D(),
    
    layers.BatchNormalization(),
    
    layers.Dropout(0.4),
    
    layers.Dense(
        len(class_names),
        activation="softmax"
    )
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    
    loss="sparse_categorical_crossentropy",
    
    metrics=["accuracy"]
)

model.summary()

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "/kaggle/working/best_skin_disease_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

In [ ]:
history = model.fit(
    train_dataset,
    
    validation_data=valid_dataset,
    
    epochs=25,
    
    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ]
)